# Step 14 -- CARE-Sim Smoke Test (Colab)

Verifies the full CARE-Sim transformer world model pipeline using synthetic data.
No real data needed -- everything is generated on the fly.

## Before running

Upload the following files to Google Drive, keeping the folder structure exactly as shown:

```
MyDrive/CareAI/
  src/
    careai/
      __init__.py
      icu_readmit/
        __init__.py
        caresim/
          __init__.py
          model.py
          dataset.py
          train.py
          ensemble.py
          simulator.py
  scripts/
    icu_readmit/
      step_14_caresim_smoke_test.py
```

All files are in your local repo under `src/careai/icu_readmit/caresim/` and `scripts/icu_readmit/`.

Then: **Runtime → Change runtime type → T4 GPU** and run all cells top to bottom.

In [ ]:
# ── Cell 1: Mount Drive + check device ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU -- running on CPU (smoke test will still work fine)')

In [ ]:
# ── Cell 2: Paths + package init files ───────────────────────────────────────
import os

DRIVE_ROOT  = '/content/drive/MyDrive/CareAI'
SCRIPT_PATH = os.path.join(DRIVE_ROOT, 'scripts', 'icu_readmit', 'step_14_caresim_smoke_test.py')

# Create __init__.py files if they were not uploaded
# (Python needs these to treat directories as packages)
for pkg_dir in [
    os.path.join(DRIVE_ROOT, 'src', 'careai'),
    os.path.join(DRIVE_ROOT, 'src', 'careai', 'icu_readmit'),
    os.path.join(DRIVE_ROOT, 'src', 'careai', 'icu_readmit', 'caresim'),
]:
    init = os.path.join(pkg_dir, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()
        print(f'Created {init}')

# Verify all required files are present
required = [
    os.path.join(DRIVE_ROOT, 'src', 'careai', 'icu_readmit', 'caresim', f)
    for f in ['model.py', 'dataset.py', 'train.py', 'ensemble.py', 'simulator.py']
] + [SCRIPT_PATH]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    print('MISSING FILES -- upload these to Drive before continuing:')
    for f in missing:
        print(f'  {f}')
    raise FileNotFoundError('Upload the missing files and re-run this cell.')
else:
    print('All required files found. Ready to run.')

In [ ]:
# ── Cell 3: Run smoke test ────────────────────────────────────────────────────
# Runs the smoke test script directly. Uses synthetic data -- no parquet needed.
# Expected output: ALL TESTS PASSED at the end.

import subprocess, sys

env = {
    **os.environ,
    'PYTHONPATH': os.path.join(DRIVE_ROOT, 'src'),
}

result = subprocess.run(
    [sys.executable, SCRIPT_PATH],
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'Smoke test FAILED (exit code {result.returncode})')
else:
    print('Smoke test completed successfully.')